# Códigos de teste

## Leitura de dados

### Auto Loader

In [0]:
%%writefile 'code/read/autoloader/01_N.py'
# Ingere dados do storage
bronzeDF = spark.readStream.format("json") \
                .load(path+"/raw/atm_visits")

# Escreve dados do stream para tabela Delta
bronzeDF.writeStream.format("delta") \
        .option("checkpointLocation", path+"/checkpoints/bronze") \
        .trigger(processingTime="10 seconds") \
        .toTable("visits_bronze")

In [0]:
%%writefile 'code/read/autoloader/02_S.py'
# Ingere dados do storage de forma incremental
bronzeDF = spark.readStream.format("cloudFiles") \
                .option("cloudFiles.format", "json") \
                .option("cloudFiles.schemaLocation", path+"/schemas") \
                .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
                .option("cloudFiles.inferColumnTypes", True) \
                .option("cloudFiles.maxFilesPerTrigger", 1) \
                .load(path+"/raw/atm_visits")

# Escreve dados do stream para tabela Delta
bronzeDF.writeStream.format("delta") \
        .option("checkpointLocation", path+"/checkpoints/bronze") \
        .trigger(processingTime="10 seconds") \
        .toTable("visits_bronze")

In [0]:
%%writefile 'code/read/autoloader/03_S.py'
# Ingere dados da tabela
bronzeDF = spark.readStream.table("cat.sch.tbl")

# Escreve dados do stream para tabela Delta
bronzeDF.writeStream.format("delta") \
        .option("checkpointLocation", path+"/checkpoints/bronze") \
        .trigger(processingTime="10 seconds") \
        .toTable("visits_bronze")

## Escrita de dados

### Delta

In [0]:
%%writefile 'code/write/delta/01_S.py'
# Sobreescreve os dados da tabela tbl com o dataframe df
df.write.mode("overwrite") \
        .saveAsTable("cat.sch.tbl")

In [0]:
%%writefile 'code/write/delta/02_S.py'
# Sobreescreve os dados da tabela tbl com os dados do dataframe df
df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("cat.sch.tbl")

In [0]:
%%writefile 'code/write/delta/03_N.py'
# Sobreescreve os dados da tabela tbl com os dados do dataframe df
df.write.format("csv") \
        .mode("overwrite") \
        .saveAsTable("cat.sch.tbl")

### Liquid Clustering / Particionamento

In [0]:
%%writefile 'code/write/liquid/01_N.py'
# Sobreescreve os dados da tabela tbl com os dados do dataframe df
df.write.mode("overwrite") \
        .partitionBy("state") \
        .saveAsTable("cat.sch.tbl")

### TODO: Expectations

## Documentação

### Código

In [0]:
%%writefile 'code/doc/code/01_N.py'
def add(a, b):
    return a + b

In [0]:
%%writefile 'code/doc/code/02_S.py'
def add(a: int, b: int) -> int:
  '''
  Soma dois números inteiros.
  
  Args:
    a (int): O primeiro número.
    b (int): O segundo número.
  
  Returns:
    int: A soma dos dois números.
  '''
  
  # Realiza a soma dos dois números inteiros
  return a + b

### TODO: Tabelas e colunas

## Boas Práticas

### TODO: Displays

### TODO: Cache

# Valida os códigos

## Preparação dos dados

### Carrega os códigos

In [0]:
import os
import pandas as pd
from pyspark.sql.functions import regexp_extract, col, pandas_udf

folder_path = os.path.abspath("code")
file_paths = []
for root, dirs, files in os.walk(folder_path):
  for file in files:
    file_paths.append(os.path.join(root, file))

@pandas_udf("string")
def read_file_udf(paths: pd.Series) -> pd.Series:
    return paths.apply(lambda p: open(p, encoding="utf-8").read())

df_code = (spark.createDataFrame([(p) for p in file_paths], ["path"])
  .withColumn("code", read_file_udf(col("path")))
  .withColumn("aval_humano", regexp_extract(col("path"), '_([^_]+)\\.py', 1))
)

display(df_code)

### Quebra em chunks

In [0]:
%pip install -qU langchain-text-splitters

In [0]:
from pyspark.sql.functions import pandas_udf, col, explode, monotonically_increasing_id
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

def split_code(code):
  python_splitter = RecursiveCharacterTextSplitter.from_language(
      language=Language.PYTHON, chunk_size=500, chunk_overlap=0
  )
  return [d.page_content for d in python_splitter.create_documents([code])]

@pandas_udf("array<string>")
def split_code_udf(code_series):
    return code_series.apply(split_code)
  
df_chunks = (df_code
  .withColumn("chunks", split_code_udf(col("code")))
  .withColumn("chunk", explode(col("chunks")))
  .select("path", "chunk", "aval_humano")
)

display(df_chunks)

## Função de revisão

In [0]:
def revisar_codigo(model, prompt, df):
  df.createOrReplaceTempView('df_chunks')
  r = spark.sql(f"select *, ai_query('{model}', concat('{prompt}', chunk)) as result from df_chunks")
  r.display()

## Leitura de dados

In [0]:
model = 'databricks-llama-4-maverick'
# model = 'databricks-claude-sonnet-4'

In [0]:
prompt = """
Você é um especialista de qualidade de código. Você deve revisar o código conforme descrito abaixo:

- Identifique se ele contém uma leitura de dados. Caso contrário, considere o critério atendido.
- Se houver uma leitura de dados, avalie se ele atende aos critérios abaixo. Use "S" quando o código atende ao critério e "N" quando não atende.
- Se o código for reprovado em ao menos um critério, explique o motivo e forneça uma sugestão de correção.

CRITÉRIOS:
1. Para leitura de arquivos no S3, deve ser utilizado o Auto Loader (cloudfiles). Não inclui leitura de tabelas.

Gere a resposta em formato JSON conforme o formato abaixo:
{
  "motivo": "<Se o código foi reprovado, explique o motivo>",
  "sugestao": "<Se o código foi reprovado, reescreva um trecho de código para exemplificar como atender os critérios que foram reprovados>",
  "criterio_01": "N",
  "criterio_02": "S",
  "criterio_03": "S",
  ...
}

Não adicione nenhum texto além do JSON, como mensagens de que é um texto gerado por IA ou pontuações (aspas, backticks, etc).

CÓDIGO:

"""

In [0]:
revisar_codigo(model, prompt, df_chunks)

## Escrita de dados

In [0]:
# model = 'databricks-llama-4-maverick'
model = 'databricks-claude-3-7-sonnet'
# model = 'databricks-claude-sonnet-4'

In [0]:
prompt = """
Você é um especialista de qualidade de código. Você deve revisar o código conforme descrito abaixo:

- Identifique se ele contém uma escrita de dados. Caso contrário, considere o critério atendido.
- Se houver uma escrita de dados, avalie se ele atende aos critérios abaixo. Use "S" quando o código atende ao critério e "N" quando não atende.
- Se o código for reprovado em ao menos um critério, explique o motivo e forneça uma sugestão de correção.

CRITÉRIOS:
1. Os métodos write.format() ou writeStream.format() não devem ser utilizados. Caso sejam utilizados, deve ser em conjunto com o parâmetro "delta".
2. Se for utilizado particionamento, recomende a substituição por Liquid Clustering (.clusterBy("col1", "col2")).

Gere a resposta em formato JSON conforme o formato abaixo:
{
  "motivo": "<Se o código foi reprovado, explique o motivo>",
  "sugestao": "<Se o código foi reprovado, reescreva um trecho de código para exemplificar como atender os critérios que foram reprovados>",
  "criterio_01": "N",
  "criterio_02": "S",
  "criterio_03": "S",
  ...
} 

Não adicione nenhum texto além do JSON, como mensagens de que é um texto gerado por IA ou pontuações (aspas, backticks, etc).

CÓDIGO:

"""

In [0]:
revisar_codigo(model, prompt, df_chunks)

## Documentação

In [0]:
# model = 'databricks-llama-4-maverick'
model = 'databricks-claude-3-7-sonnet'
# model = 'databricks-claude-sonnet-4'

In [0]:
prompt = """
Você é um especialista de qualidade de código. Você deve revisar o código conforme descrito abaixo:

- Avalie se ele atende aos critérios abaixo. Use "S" quando o código atende ao critério e "N" quando não atende.
- Se o código for reprovado em ao menos um critério, explique o motivo e forneça uma sugestão de correção.

CRITÉRIOS:
1. Comentário simples com uma descrição do objetivo do bloco de código. Não é necessário fornecer maiores detalhes, como formato dos dados, tipos de dados, propósito, etc.
2. Não é necessário conter uma declaração de função explícita, mas, se houver, esta deve conter:
  - Docstring com descrição da função
  - Variáveis de entrada e saída devem conter tipo e descrição

Gere a resposta em formato JSON conforme o formato abaixo:
{
  "motivo": "<Se o código foi reprovado, explique o motivo>",
  "sugestao": "<Se o código foi reprovado, reescreva um trecho de código para exemplificar como atender os critérios que foram reprovados>",
  "criterio_01": "N",
  "criterio_02": "S",
  "criterio_03": "S",
  ...
} 

Não adicione nenhum texto além do JSON, como mensagens de que é um texto gerado por IA ou pontuações (aspas, backticks, etc).

CÓDIGO:

"""

In [0]:
revisar_codigo(model, prompt, df_chunks)

In [0]:
# df_code = spark.createDataFrame([(c) for c in code], ["code"])
# display(df_code)

In [0]:
# code = code.replace("\'","\\'")

In [0]:
# %md # Prompt Geral

# 3. O código segue as melhores práticas de programação.
# 4. O código é seguro e evita problemas de segurança.
# 5. O código é compatível com as especificações do projeto.

In [0]:
# prompt = """
# Você é um especialista de qualidade de código. Você deve revisar o código para:

# - Identificar se ele contém os itens descritos abaixo. Caso o código não contenha um item, considere o critério atendido.
# - Caso o item esteja presente no código, avaliar se ele atende ao critério. Use "S" quando o código atende ao critério e "N" quando não atende.

# 1. O código deve possuir documentação, incluindo:
#   - Comentários explicando o objetivo do código
#   - Se houverem declarações de funções:
#     - Esta função deve conter descrição, tipos e descrições das variáveis de entrada e saída.

# 2. Leitura de dados:
#   - Para leitura de arquivos no S3, deve ser utilizado o Auto Loader (cloudfiles)

# 3. Escrita de dados:
#   - O método .format() não deve ser utilizado. Caso seja utilizado, deve ser feito em conjunto com o parâmetro "delta"
#   - Se for utilizado particionamento, recomende a substituição por Liquid Clustering (.clusterBy("col1", "col2"))

# Gere a resposta em formato JSON conforme o formato abaixo:
# {
#   "resumo": "<Explique porque o código foi reprovado>",
#   "sugestao": "<Reescreva um trecho de código para exemplificar como atender os critérios que foram reprovados>",
#   "criterio_01": "N",
#   "criterio_02": "S",
#   "criterio_03": "S",
#   ...
# } 

# Não adicione nenhum texto além do JSON, como mensagens de que é um texto gerado por IA ou pontuações (aspas, backticks, etc).

# CÓDIGO:

# """

In [0]:
# def revisar_codigo(model, prompt, df):
#   df.createOrReplaceTempView('df_chunks')
#   r = (spark.sql(f"""
#       select *,
#         from_json(
#           ai_query(
#             '{model}',
#             concat('{prompt}', chunk)
#           ),
#           'STRUCT<motivo:string, sugestao:string, criterio_01:string>'
#         ) as result
#       from df_chunks
#     """)
#     .selectExpr("id", "chunk", "result.*")
#   )

#   r.display()